# LangChain Model Switcher - AI Math Assistant

This notebook demonstrates how to build an AI math assistant that can work with multiple LLM providers:
- IBM Watson (Granite models)
- Anthropic Claude
- Ollama (local models)

Switch between models by setting the `MODEL_PROVIDER` environment variable!

## Setup and Configuration

In [1]:
# Enable auto-reload for development
%load_ext autoreload
%autoreload 2

import os
import sys
from pathlib import Path

# 🔧 THE KEY FIX: Add PROJECT ROOT (not src) to path
project_root = Path.cwd().parent if 'notebooks' in str(Path.cwd()) else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print(f"✅ Project root: {project_root}")
print(f"📁 Current working directory: {Path.cwd()}")
print(f"🐍 Python path includes project root: {str(project_root) in sys.path}")

✅ Project root: /Users/family_crawfords/projects/claude-mcp/langchain-model-switcher
📁 Current working directory: /Users/family_crawfords/projects/claude-mcp/langchain-model-switcher/notebooks
🐍 Python path includes project root: True


In [2]:
# 🎯 THE KEY FIX: Import from 'src' as a package
# This works because we added PROJECT ROOT to path, so Python sees 'src' as a package
from src.utils.model_factory import ModelFactory, get_model
from src.config.settings import get_current_provider, get_model_config
from src.mcp.tools import get_math_tools

# Set the model provider - change this to switch models!
# Options: \"watson\", \"claude\", \"ollama\" etc.
MODEL_PROVIDER = "gpt_oss_20b"  # Change this to test different models
os.environ["MODEL_PROVIDER"] = MODEL_PROVIDER

print(f"🤖 Using model provider: {MODEL_PROVIDER}")
print(f"📋 Available providers: {ModelFactory.list_available_providers()}")

🤖 Using model provider: gpt_oss_20b
📋 Available providers: ['watson', 'claude', 'ollama', 'phi', 'openai', 'gpt_oss_20b']


## Model Initialization

The beauty of our model switcher is that this single line works with any provider!

In [3]:
# Get the current model - this works with any provider!
model_adapter = get_model()
llm = model_adapter.get_langchain_model()

print(f"🎯 Model adapter: {model_adapter}")
print(f"🏷️ Model name: {model_adapter.get_model_name()}")
print(f"🛠️ Supports tool calling: {model_adapter.supports_tool_calling()}")
print(f"⚙️ LangChain model: {type(llm).__name__}")

🎯 Model adapter: GPTAdapter(gpt-oss:20b:20b)
🏷️ Model name: gpt-oss:20b:20b
🛠️ Supports tool calling: True
⚙️ LangChain model: ChatOllama


## Test Basic Model Functionality

In [4]:
# Test the model with a simple query
response = llm.invoke("What is tool calling in LangChain? Answer in 2 sentences.")
print("🤖 Model Response:")
print(response.content)

INFO:httpx:HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"


🤖 Model Response:
Tool calling in LangChain is a feature that lets a language model dynamically invoke external functions or APIs—called “tools”—based on the conversation context, enabling it to perform tasks like data retrieval, calculations, or database queries. By integrating these tools, the model can produce more accurate, up‑to‑date, and actionable responses while still maintaining a natural dialogue flow.


In [ ]:
# Test the model with a simple query
response = llm.invoke("What is tool calling in LangChain? Answer in 2 sentences.")
print("🤖 Model Response:")
print(response.content)

INFO:httpx:HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"


🤖 Model Response:
In LangChain, a "tool" refers to a specific AI model or algorithm that can be used to generate text, answer questions, or perform other language-related tasks. By integrating multiple tools into a single interface, LangChain enables users to easily switch between different models and techniques to achieve their desired outcomes in natural language processing (NLP) applications.


## Load Mathematical Tools

These tools work with any model provider through our unified interface.

In [5]:
# Get all mathematical tools
tools = get_math_tools()

print("🛠️ Available tools:")
for tool in tools:
    print(f"  - {tool.name}: {tool.description}")

🛠️ Available tools:
  - add_numbers: Adds a list of numbers provided in the input string.

Parameters:
- inputs (str): string containing numbers that can be extracted and summed.

Returns:
- dict: A dictionary with a single key "result" containing the sum of the numbers.

Example Input: "Add the numbers 10, 20, and 30."
Example Output: {"result": 60}
  - subtract_numbers: Extracts numbers from a string and performs subtraction sequentially, starting with the first number.

Parameters:
- inputs (str): A string containing numbers to subtract.

Returns:
- dict: A dictionary containing the key "result" with the calculated difference.

Example Input: "100, 20, 10"
Example Output: {"result": 70}
  - multiply_numbers: Extracts numbers from a string and calculates their product.

Parameters:
- inputs (str): A string containing numbers separated by spaces, commas, or other delimiters.

Returns:
- dict: A dictionary with the key "result" containing the product of the numbers.

Example Input: "2,

## Create the Math Agent

This agent works seamlessly with any model provider!

In [6]:
from langgraph.prebuilt import create_react_agent

# Create the agent with all tools
math_agent = create_react_agent(
    model=llm,  # This now works with any provider!
    tools=tools,
    prompt="You are a helpful mathematical assistant that can perform various operations and look up information. Use the tools precisely and explain your reasoning clearly."
)

print(f"🤖 Math agent created with {len(tools)} tools")
print(f"⚙️ Agent is using: {model_adapter.get_model_name()}")

🤖 Math agent created with 5 tools
⚙️ Agent is using: gpt-oss:20b


## Test Mathematical Operations

In [7]:
# Test addition
response = math_agent.invoke({
    "messages": [("human", "Add the numbers 25, 15, and 10")]
})

print("➕ Addition Test:")
print(f"Query: Add the numbers 25, 15, and 10")
print(f"Answer: {response['messages'][-1].content}")

INFO:httpx:HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"


➕ Addition Test:
Query: Add the numbers 25, 15, and 10
Answer: Sure! Adding the numbers together:

- 25 + 15 = 40  
- 40 + 10 = 50  

**Result:** 50


In [8]:
# Test multiplication
response = math_agent.invoke({
    "messages": [("human", "Multiply 6, 7, and 2")]
})

print("✖️ Multiplication Test:")
print(f"Query: Multiply 6, 7, and 2")
print(f"Answer: {response['messages'][-1].content}")

INFO:httpx:HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"


✖️ Multiplication Test:
Query: Multiply 6, 7, and 2
Answer: The product of 6, 7, and 2 is **84**.


In [9]:
# Test division
response = math_agent.invoke({
    "messages": [("human", "Divide 144 by 12 and then by 3")]
})

print("➗ Division Test:")
print(f"Query: Divide 144 by 12 and then by 3")
print(f"Answer: {response['messages'][-1].content}")

INFO:httpx:HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"


➗ Division Test:
Query: Divide 144 by 12 and then by 3
Answer: First divide 144 by 12:

\[
144 \div 12 = 12
\]

Then divide that result by 3:

\[
12 \div 3 = 4
\]

So the final answer is **4**.


In [10]:
# Test subtraction
response = math_agent.invoke({
    "messages": [("human", "Subtract 30 and 15 from 100")]
})

print("➖ Subtraction Test:")
print(f"Query: Subtract 30 and 15 from 100")
print(f"Answer: {response['messages'][-1].content}")

INFO:httpx:HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"


➖ Subtraction Test:
Query: Subtract 30 and 15 from 100
Answer: You want to subtract 30 and 15 from 100, which is calculated as:

\[
100 - 30 - 15 = 55
\]

So the result is **55**.


## Test Complex Multi-Step Operations

In [11]:
# Test complex operation
complex_query = "Calculate (25 + 15) multiplied by 3, then subtract 20"

response = math_agent.invoke({
    "messages": [("human", complex_query)]
})

print("🧮 Complex Operation Test:")
print(f"Query: {complex_query}")
print(f"Answer: {response['messages'][-1].content}")

INFO:httpx:HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"


🧮 Complex Operation Test:
Query: Calculate (25 + 15) multiplied by 3, then subtract 20
Answer: First, add the numbers inside the parentheses:

\(25 + 15 = 40\)

Next, multiply the result by 3:

\(40 \times 3 = 120\)

Finally, subtract 20:

\(120 - 20 = 100\)

**Result: 100**


## Test Wikipedia Integration

In [12]:
# Test Wikipedia search with mathematical calculation
wiki_math_query = "What is the population of Canada? Then multiply it by 0.25"

response = math_agent.invoke({
    "messages": [("human", wiki_math_query)]
})

print("🌐 Wikipedia + Math Test:")
print(f"Query: {wiki_math_query}")
print(f"Answer: {response['messages'][-1].content}")

INFO:httpx:HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"


🌐 Wikipedia + Math Test:
Query: What is the population of Canada? Then multiply it by 0.25
Answer: **Population of Canada (2025 estimate)**  
- Approximately **41.5 million** people.

**Multiplying by 0.25**

\[
41.5\ \text{million} \times 0.25 = 10.375\ \text{million}
\]

So, 25 % of Canada’s population is about **10.4 million** people.


## Model Comparison Demo

Let's test the same query with different models to see how they perform!

In [14]:
def test_model_with_query(provider: str, query: str):
    """Test a specific model provider with a query."""
    try:
        # Set the provider
        os.environ["MODEL_PROVIDER"] = provider
        
        # Get the model
        adapter = get_model(provider)
        model = adapter.get_langchain_model()
        
        # Create agent
        agent = create_react_agent(
            model=model,
            tools=tools,
            prompt="You are a helpful mathematical assistant."
        )
        
        # Test the query
        response = agent.invoke({"messages": [("human", query)]})
        
        return {
            "provider": provider,
            "model_name": adapter.get_model_name(),
            "success": True,
            "response": response['messages'][-1].content
        }
    except Exception as e:
        return {
            "provider": provider,
            "model_name": "N/A",
            "success": False,
            "error": str(e)
        }

# Test query
test_query = "Add 15 and 25, then multiply by 2"

print(f"🧪 Testing query: '{test_query}'\n")
print("=" * 80)

# Test each available provider
providers_to_test = ["phi", "ollama", "gpt_oss_20b"]

for provider in providers_to_test:
    print(f"\n🔍 Testing {provider.upper()}:")
    print("-" * 40)
    
    result = test_model_with_query(provider, test_query)
    
    if result["success"]:
        print(f"✅ Model: {result['model_name']}")
        print(f"Response: {result['response']}")
    else:
        print(f"❌ Error with {provider}: {result['error']}")

# Reset to original provider
os.environ["MODEL_PROVIDER"] = MODEL_PROVIDER

🧪 Testing query: 'Add 15 and 25, then multiply by 2'


🔍 Testing PHI:
----------------------------------------


INFO:httpx:HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"


✅ Model: Phi-llama3.2
Response: To calculate the result of adding 15 and 25, then multiplying by 2:

First, add 15 and 25: 
15 + 25 = 40

Then, multiply the result by 2:
40 * 2 = 80

🔍 Testing OLLAMA:
----------------------------------------


INFO:httpx:HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"


✅ Model: Ollama-llama3.1
Response: To add 15 and 25, we get:

15 + 25 = 40

Then, multiplying by 2 gives us:

40 × 2 = 80

So the final result is 80.

🔍 Testing GPT_OSS_20B:
----------------------------------------


INFO:httpx:HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"


✅ Model: gpt-oss:20b
Response: The result is **80**.


## Quick Provider Switch Demo

See how easy it is to switch between models!

In [15]:
# Function to quickly switch and test
def quick_switch_test(new_provider: str):
    """Quickly switch provider and test with a simple calculation."""
    os.environ["MODEL_PROVIDER"] = new_provider
    
    try:
        adapter = get_model()
        print(f"✅ Switched to: {adapter.get_model_name()}")
        
        # Quick test
        llm_test = adapter.get_langchain_model()
        response = llm_test.invoke("What is 5 + 3? Answer with just the number.")
        print(f"🧮 Quick test response: {response.content}")
        
    except Exception as e:
        print(f"❌ Error switching to {new_provider}: {e}")

print("🔄 Testing provider switching:")
print("\n1. Testing Small Llama:")
quick_switch_test("phi")

print("\n2. Testing GPT:")
quick_switch_test("gpt_oss_20b")

print("\n3. Testing Ollama (if available):")
quick_switch_test("ollama")

# Reset to original
os.environ["MODEL_PROVIDER"] = MODEL_PROVIDER
print(f"\n🔄 Reset to: {MODEL_PROVIDER}")

🔄 Testing provider switching:

1. Testing Small Llama:
✅ Switched to: Phi-llama3.2


INFO:httpx:HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"


🧮 Quick test response: 8

2. Testing GPT:
✅ Switched to: gpt-oss:20b


INFO:httpx:HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"


🧮 Quick test response: 8

3. Testing Ollama (if available):
✅ Switched to: Ollama-llama3.1


INFO:httpx:HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"


🧮 Quick test response: 8

🔄 Reset to: gpt_oss_20b


## Vector Database Applications

Your chunk store applications can now use any model! Here's a preview of how you'd integrate with a vector database:

In [16]:
# Example of how to use with vector databases
# This demonstrates the concept - actual implementation would require vector store setup

print("🗄️ Vector Database Integration Example:")
print("===================================")
print()
print("# Example code for vector database integration:")
print()
print("from langchain_community.vectorstores import Chroma")
print("from langchain.embeddings import OpenAIEmbeddings")
print("from langchain.text_splitter import RecursiveCharacterTextSplitter")
print("from src.utils.model_factory import get_model")
print()
print("# Get any model for your RAG system")
print("llm = get_model().get_langchain_model()")
print()
print("# Set up vector store (works with any model!)")
print("embeddings = OpenAIEmbeddings()")
print("vectorstore = Chroma(embedding_function=embeddings)")
print()
print("# Create retrieval chain")
print("from langchain.chains import RetrievalQA")
print("qa_chain = RetrievalQA.from_chain_type(")
print("    llm=llm,  # Any model works here!")
print("    chain_type='stuff',")
print("    retriever=vectorstore.as_retriever()")
print(")")
print()
print("# Switch models anytime:")
print("# os.environ['MODEL_PROVIDER'] = 'ollama'  # or 'phi'")
print("# llm = get_model().get_langchain_model()")
print("# qa_chain.llm = llm  # Update the chain with new model")

current_model = get_model()
print(f"\n🎯 Currently configured for: {current_model.get_model_name()}")
print(f"🛠️ This model {'✅ supports' if current_model.supports_tool_calling() else '❌ does not support'} tool calling")

🗄️ Vector Database Integration Example:

# Example code for vector database integration:

from langchain_community.vectorstores import Chroma
from langchain.embeddings import OpenAIEmbeddings
from langchain.text_splitter import RecursiveCharacterTextSplitter
from src.utils.model_factory import get_model

# Get any model for your RAG system
llm = get_model().get_langchain_model()

# Set up vector store (works with any model!)
embeddings = OpenAIEmbeddings()
vectorstore = Chroma(embedding_function=embeddings)

# Create retrieval chain
from langchain.chains import RetrievalQA
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,  # Any model works here!
    chain_type='stuff',
    retriever=vectorstore.as_retriever()
)

# Switch models anytime:
# os.environ['MODEL_PROVIDER'] = 'ollama'  # or 'phi'
# llm = get_model().get_langchain_model()
# qa_chain.llm = llm  # Update the chain with new model

🎯 Currently configured for: gpt-oss:20b
🛠️ This model ✅ supports tool calling


## Summary

🎉 **Congratulations!** You've successfully created a unified LangChain interface that works with multiple LLM providers:

### Key Benefits:
- **🔄 Easy Switching**: Change models with one environment variable
- **🧩 Unified Interface**: Same code works with Watson, Claude, and Ollama
- **🛠️ MCP Ready**: Tools are exposed via Model Context Protocol
- **📈 Scalable**: Easy to add new model providers
- **🏗️ Vector Database Ready**: Perfect foundation for RAG applications

### Next Steps:
1. **Vector Database Integration**: Build RAG systems with your chunk store
2. **Custom Tools**: Add domain-specific tools for your use case
3. **MCP Server**: Deploy the MCP server for external access
4. **Production Setup**: Configure API keys and deploy

### Model Switching Commands:
```bash
# Switch to Claude
export MODEL_PROVIDER=claude

# Switch to Watson
export MODEL_PROVIDER=watson

# Switch to Ollama
export MODEL_PROVIDER=ollama
```

Your chunk store applications will work seamlessly with any of these models! 🚀